# Analisis Data Penjualan Apotek
## Tahap 1 — Import & Strukturisasi Data

**Sumber data:** `sales.sql` — dump database `db_pharmacy` (MariaDB)
**Periode:** Januari – Desember 2015
**Tabel:** `ms_product`, `ms_sales`, `det_sales`, `transaction`

---

Notebook ini mencakup **tiga tahap** pengolahan data:
- **[TAHAP 1]** Import & Strukturisasi <- *sedang dikerjakan*
- **[TAHAP 2]** Cleaning & Transformasi
- **[TAHAP 3]** EDA & Ringkasan Awal


In [1]:
# Install jika belum ada: !pip install pandas

import re
import json
import warnings
from collections import defaultdict
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'pandas  : {pd.__version__}')
print('Library siap.')


pandas  : 2.2.2
Library siap.


---
## 1.1 Setup Environment


In [2]:
BASE_DIR   = Path(r"d:\PORTOFOLIO\sales")
SQL_FILE   = BASE_DIR / "data/sales.sql"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_TABLES = ['ms_product', 'ms_sales', 'det_sales', 'transaction']

assert SQL_FILE.exists(), f"File tidak ditemukan: {SQL_FILE}"

file_size_mb = SQL_FILE.stat().st_size / (1024 ** 2)
print(f"File    : {SQL_FILE.name}")
print(f"Ukuran  : {file_size_mb:.1f} MB")
print(f"Output  : {OUTPUT_DIR}")


File    : sales.sql
Ukuran  : 67.3 MB
Output  : d:\PORTOFOLIO\sales\output


---
## 1.2 Inspeksi Struktur SQL File


In [3]:
# Peek 50 baris pertama untuk melihat format dump
sep = '=' * 64
print(sep)
print(f"50 BARIS PERTAMA: {SQL_FILE.name}")
print(sep)

with open(SQL_FILE, 'r', encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 50:
            break
        print(f"{i+1:3}: {line}", end='')


50 BARIS PERTAMA: sales.sql
  1: -- --------------------------------------------------------
  2: -- Host:                         127.0.0.1
  3: -- Server version:               10.4.6-MariaDB - mariadb.org binary distribution
  4: -- Server OS:                    Win64
  5: -- HeidiSQL Version:             11.3.0.6295
  6: -- --------------------------------------------------------
  7: 
  8: /*!40101 SET @OLD_CHARACTER_SET_CLIENT=@@CHARACTER_SET_CLIENT */;
  9: /*!40101 SET NAMES utf8 */;
 10: /*!50503 SET NAMES utf8mb4 */;
 11: /*!40014 SET @OLD_FOREIGN_KEY_CHECKS=@@FOREIGN_KEY_CHECKS, FOREIGN_KEY_CHECKS=0 */;
 12: /*!40101 SET @OLD_SQL_MODE=@@SQL_MODE, SQL_MODE='NO_AUTO_VALUE_ON_ZERO' */;
 13: /*!40111 SET @OLD_SQL_NOTES=@@SQL_NOTES, SQL_NOTES=0 */;
 14: 
 15: -- Dumping structure for table db_pharmacy.det_sales
 16: CREATE TABLE IF NOT EXISTS `det_sales` (
 17:   `NO_RESEP` varchar(20) CHARACTER SET latin1 NOT NULL,
 18:   `KD_OBAT` varchar(20) CHARACTER SET latin1 NOT NULL,
 19:

In [4]:
# Deteksi tabel & format INSERT (dengan/tanpa daftar kolom)
found_tables = []
insert_samples = {}    # simpan 1 contoh baris INSERT per tabel

with open(SQL_FILE, 'r', encoding='utf-8', errors='replace') as f:
    for line in f:
        stripped = line.strip()

        # Deteksi CREATE TABLE
        m = re.match(r'CREATE TABLE[^`]*`(\w+)`', stripped)
        if m:
            found_tables.append(m.group(1))

        # Deteksi format INSERT (simpan 1 sample per tabel)
        ins = re.match(r'INSERT INTO `?(\w+)`?\s*(.*?)VALUES', stripped,
                       re.IGNORECASE)
        if ins:
            tbl  = ins.group(1)
            cols = ins.group(2).strip()
            if tbl not in insert_samples:
                insert_samples[tbl] = {
                    'has_col_list': bool(re.match(r'\(', cols)),
                    'preview': stripped[:100]
                }

print(f"Tabel ditemukan ({len(found_tables)}):")
for t in found_tables:
    tag  = '[TARGET]' if t in TARGET_TABLES else '       '
    info = insert_samples.get(t, {})
    col_tag = '[+col_list]' if info.get('has_col_list') else '[values_only]'
    print(f"  {tag}  {t:<20}  INSERT format: {col_tag}")
    if info:
        print(f"             preview: {info['preview'][:80]}")


Tabel ditemukan (4):
  [TARGET]  det_sales             INSERT format: [+col_list]
             preview: INSERT INTO `det_sales` (`NO_RESEP`, `KD_OBAT`, `QTY`, `HNA`, `HJ`, `PPN_JUAL`) 
  [TARGET]  ms_product            INSERT format: [+col_list]
             preview: INSERT INTO `ms_product` (`KD_OBAT`, `NAMA`, `SAT_JUAL`, `KD_PABRIK`, `HJ_RP`) V
  [TARGET]  ms_sales              INSERT format: [+col_list]
             preview: INSERT INTO `ms_sales` (`NO_RESEP`, `TGL`, `KD_CUST`, `KD_DOKTER`, `REG_AS`, `JA
  [TARGET]  transaction           INSERT format: [+col_list]
             preview: INSERT INTO `transaction` (`NO_RESEP`, `TGL`, `KD_CUST`, `KD_OBAT`, `QTY`, `HNA`


---
## 1.3 Parse SQL Dump -> DataFrames

Parser streaming baris per baris.
INSERT format yang didukung:
- `INSERT INTO \`tabel\` VALUES (...)` ← tanpa daftar kolom
- `INSERT INTO \`tabel\` (\`col1\`,\`col2\`) VALUES (...)` ← dengan daftar kolom


In [5]:
def _parse_values(values_str):
    """
    Parse SQL VALUES string -> list of lists (state machine).
    Menangani: NULL->None, string literal, escape backslash & ''.
    """
    rows        = []
    current_row = []
    current_val = []
    in_quote    = False
    paren_depth = 0
    i = 0
    n = len(values_str)

    while i < n:
        c = values_str[i]

        if in_quote:
            # escape: backslash diikuti karakter apa pun
            if c == chr(92) and i + 1 < n:
                current_val.append(c)
                current_val.append(values_str[i + 1])
                i += 2
                continue
            elif c == "'":
                # '' = escaped single-quote dalam SQL
                if i + 1 < n and values_str[i + 1] == "'":
                    current_val.append("'")
                    i += 2
                    continue
                in_quote = False
            else:
                current_val.append(c)
        else:
            if   c == "'": in_quote = True
            elif c == '(':
                paren_depth += 1
            elif c == ')':
                paren_depth -= 1
                if paren_depth == 0:
                    val = ''.join(current_val).strip()
                    current_row.append(None if val == 'NULL' else val)
                    rows.append(current_row)
                    current_row = []
                    current_val = []
            elif c == ',':
                if paren_depth == 1:
                    val = ''.join(current_val).strip()
                    current_row.append(None if val == 'NULL' else val)
                    current_val = []
            elif paren_depth == 1:
                current_val.append(c)
        i += 1

    return rows


print("_parse_values siap.")


_parse_values siap.


In [6]:
def parse_sql_to_dataframes(filepath, target_tables=None, encoding='utf-8'):
    """
    Parse SQL dump MariaDB/MySQL -> dict of pd.DataFrames (streaming).

    Mendukung dua format INSERT:
    1. Single-line : INSERT INTO `t` VALUES (...),(...);
    2. Multi-line  : INSERT INTO `t` (`c1`,`c2`) VALUES
                         ('r1v1', 'r1v2'),
                         ('r2v1', 'r2v2');   <- format HeidiSQL
    """
    filepath           = Path(filepath)
    table_columns      = {}
    table_rows         = defaultdict(list)
    current_table      = None
    in_create          = False
    current_insert_tbl = None   # state: sedang di dalam multi-line INSERT block
    total_lines        = 0
    insert_lines       = 0

    print(f"Parsing: {filepath.name}  ({filepath.stat().st_size/(1024**2):.1f} MB)")
    print(f"Target : {target_tables or 'semua tabel'}\n")

    SKIP_KW = ('KEY ', 'INDEX ', 'UNIQUE ', 'PRIMARY ', 'CONSTRAINT ', 'CHECK ')

    # Regex INSERT header — backtick & col_list opsional
    INSERT_HDR = re.compile(
        r"INSERT\s+INTO\s+`?(\w+)`?\s*"   # nama tabel
        r"(?:\([^)]*\)\s*)?"               # daftar kolom opsional
        r"VALUES\s*(.*?)(?:;)?\s*$",       # VALUES + data opsional di baris sama
        re.IGNORECASE | re.DOTALL
    )

    with open(filepath, 'r', encoding=encoding, errors='replace') as f:
        for line in f:
            total_lines += 1
            stripped = line.strip()

            # Skip baris kosong & komentar
            if not stripped or stripped.startswith('--') or stripped.startswith('/*'):
                continue

            # ── CREATE TABLE ──────────────────────────────────────────────────
            m = re.match(r'CREATE TABLE[^`]*`(\w+)`', stripped)
            if m:
                current_insert_tbl = None   # tutup INSERT block sebelumnya
                current_table = m.group(1)
                table_columns[current_table] = []
                in_create = True
                continue

            if in_create:
                if stripped.startswith(')'):
                    in_create     = False
                    current_table = None
                    continue
                col_m = re.match(r'`(\w+)`', stripped)
                if col_m and not any(stripped.upper().startswith(kw) for kw in SKIP_KW):
                    table_columns[current_table].append(col_m.group(1))
                continue

            # ── INSERT header ─────────────────────────────────────────────────
            hdr = INSERT_HDR.match(stripped)
            if hdr:
                tbl  = hdr.group(1)
                rest = (hdr.group(2) or '').strip().rstrip(';').strip()

                if target_tables and tbl not in target_tables:
                    current_insert_tbl = None
                    continue

                insert_lines += 1
                current_insert_tbl = tbl

                # Ada data inline di baris sama dengan VALUES? (single-line INSERT)
                if rest:
                    table_rows[tbl].extend(_parse_values(rest))

                # Apakah INSERT selesai di baris ini?
                if stripped.rstrip().endswith(';'):
                    current_insert_tbl = None
                continue

            # ── Data rows (multi-line INSERT) ─────────────────────────────────
            if current_insert_tbl is not None:
                tbl     = current_insert_tbl
                is_last = stripped.endswith(';')

                # Setiap baris data diawali dengan '('
                if stripped.startswith('('):
                    data = stripped.rstrip(';').rstrip(',').strip()
                    if data:
                        table_rows[tbl].extend(_parse_values(data))

                if is_last:
                    current_insert_tbl = None

    # ── Build DataFrames ──────────────────────────────────────────────────────
    print(f"Parsing selesai!")
    print(f"  Baris dibaca    : {total_lines:,}")
    print(f"  INSERT diproses : {insert_lines:,}\n")

    dataframes = {}
    for tbl in (target_tables or list(table_rows.keys())):
        rows = table_rows.get(tbl, [])
        cols = table_columns.get(tbl, [])

        if not rows:
            print(f"  [!] {tbl}: TIDAK ADA DATA — cek nama & format INSERT")
            continue

        n_data = len(rows[0])
        if len(cols) != n_data:
            print(f"  [!] {tbl}: kolom tidak cocok "
                  f"(schema={len(cols)}, data={n_data}) -> auto-index")
            cols = [f'col_{i}' for i in range(n_data)]

        df = pd.DataFrame(rows, columns=cols)
        dataframes[tbl] = df
        print(f"  [OK] {tbl:<22}: {len(df):>10,} baris x {len(df.columns)} kolom")

    return dataframes


print("parse_sql_to_dataframes siap.")


parse_sql_to_dataframes siap.


In [7]:
%%time
dfs = parse_sql_to_dataframes(
    filepath      = SQL_FILE,
    target_tables = TARGET_TABLES,
    encoding      = 'utf-8'
)


Parsing: sales.sql  (67.3 MB)
Target : ['ms_product', 'ms_sales', 'det_sales', 'transaction']

Parsing selesai!
  Baris dibaca    : 1,048,489
  INSERT diproses : 106

  [OK] ms_product            :      6,878 baris x 5 kolom
  [OK] ms_sales              :    127,064 baris x 7 kolom
  [OK] det_sales             :    514,620 baris x 6 kolom
  [OK] transaction           :    399,738 baris x 7 kolom
CPU times: total: 15.2 s
Wall time: 17.2 s


In [8]:
# ── Validasi hasil parser ────────────────────────────────────────────────────
print("Tabel yang berhasil di-parse:\n")
print(f"{'Nama':<22} {'Baris':>10} {'Kolom':>7} {'Memory':>10}  Status")
print('-' * 70)

all_ok = True
for tbl in TARGET_TABLES:
    df = dfs.get(tbl)
    if df is None:
        print(f"{tbl:<22} {'GAGAL':>10}  -- tabel tidak ditemukan di dump!")
        all_ok = False
    else:
        mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
        print(f"{tbl:<22} {len(df):>10,} {len(df.columns):>7} {mem:>8.1f} MB  OK")

print()
if not all_ok:
    print("[!] Ada tabel yang gagal di-parse. Periksa output sel di atas.")
    print("    Kemungkinan: nama tabel berbeda atau format INSERT tidak dikenali.")
else:
    print("Semua tabel berhasil di-parse.")

# Assign shorthand variables
ms_product  = dfs.get('ms_product')
ms_sales    = dfs.get('ms_sales')
det_sales   = dfs.get('det_sales')
transaction = dfs.get('transaction')


Tabel yang berhasil di-parse:

Nama                        Baris   Kolom     Memory  Status
----------------------------------------------------------------------
ms_product                  6,878       5      1.8 MB  OK
ms_sales                  127,064       7     47.6 MB  OK
det_sales                 514,620       6    161.3 MB  OK
transaction               399,738       7    151.0 MB  OK

Semua tabel berhasil di-parse.


---
## 1.4 Pemetaan Relasi Antar Tabel

| Tabel | Key | Peran |
|---|---|---|
| `ms_product` | `KD_OBAT` (PK) | Master obat/produk |
| `ms_sales` | `NO_RESEP` (PK) | Header transaksi resep |
| `det_sales` | `NO_RESEP` (FK), `KD_OBAT` (FK) | Detail item per resep |
| `transaction` | `NO_RESEP`, `KD_OBAT` | Tabel gabungan — status diverifikasi |

```
ms_product (KD_OBAT) <---- det_sales.KD_OBAT
                                |
                                | NO_RESEP (FK)
                                v
                           ms_sales (NO_RESEP PK)

transaction ---- (NO_RESEP + KD_OBAT)
               -> subset / superset / duplikat?
```


In [9]:
# Detail struktur setiap tabel
for name, df in dfs.items():
    print(f"\n{'='*64}")
    print(f"  TABEL: {name}")
    print(f"{'='*64}")
    print(f"  Shape : {df.shape[0]:,} baris x {df.shape[1]} kolom")
    print(f"\n  Kolom :")
    for col in df.columns:
        n_null = df[col].isna().sum()
        nn     = df[col].dropna()
        sample = str(nn.iloc[0]) if len(nn) > 0 else 'N/A'
        print(f"    |- {col:<24} null={n_null:>8,}  sample: {sample[:35]}")
    print(f"\n  Preview (3 baris):")
    print(df.head(3).to_string(index=False))



  TABEL: ms_product
  Shape : 6,878 baris x 5 kolom

  Kolom :
    |- KD_OBAT                  null=       0  sample: AI-0001
    |- NAMA                     null=       0  sample: Vagizol Ovula 500mg
    |- SAT_JUAL                 null=       0  sample: 45%,
    |- KD_PABRIK                null=       0  sample: suppos
    |- HJ_RP                    null=       0  sample: 0

  Preview (3 baris):
KD_OBAT                NAMA SAT_JUAL KD_PABRIK  HJ_RP
AI-0001 Vagizol Ovula 500mg     45%,    suppos      0
AI-0002 4-Epeedo 10 mg/vial     vial      Kifa 184000
AI-0003 4-Epeedo 50 mg/vial     vial      Kifa 876750

  TABEL: ms_sales
  Shape : 127,064 baris x 7 kolom

  Kolom :
    |- NO_RESEP                 null=       0  sample: B-04.2015-01-0001
    |- TGL                      null=       0  sample: 2015-04-02
    |- KD_CUST                  null=       0  sample: UMUM
    |- KD_DOKTER                null=       0  sample: 000
    |- REG_AS                   null=       0  sample: R
  

---
## 1.5 Verifikasi Integritas Data

| # | Aspek | Yang dicek |
|---|---|---|
| CEK 1 | NO_RESEP | Orphan `det_sales` <-> `ms_sales` |
| CEK 2 | KD_OBAT | Orphan `det_sales`/`transaction` <-> `ms_product` |
| CEK 3 | Coverage | Apakah `transaction` redundan / superset `ms_sales`? |
| CEK 4 | Format | Konsistensi format string `NO_RESEP` |
| CEK 5 | Qty | Qty negatif (indikasi retur/pembatalan) |


In [10]:
# ── CEK 1: NO_RESEP — det_sales <-> ms_sales ─────────────────────────────────
assert ms_sales  is not None, "ms_sales belum ter-parse — jalankan ulang sel parser"
assert det_sales is not None, "det_sales belum ter-parse — jalankan ulang sel parser"

print("=" * 64)
print("CEK 1: NO_RESEP — det_sales <-> ms_sales")
print("=" * 64)

resep_ms  = set(ms_sales['NO_RESEP'].dropna().unique())
resep_det = set(det_sales['NO_RESEP'].dropna().unique())

orphan_det = resep_det - resep_ms   # di det_sales, tidak ada di ms_sales
orphan_ms  = resep_ms  - resep_det  # di ms_sales, tidak ada di det_sales

print(f"  NO_RESEP unik di ms_sales  : {len(resep_ms):>10,}")
print(f"  NO_RESEP unik di det_sales : {len(resep_det):>10,}")
print(f"  Orphan di det_sales        : {len(orphan_det):>10,}  <- tidak ada header di ms_sales")
print(f"  Orphan di ms_sales         : {len(orphan_ms):>10,}  <- tidak ada detail di det_sales")

if orphan_det:
    print(f"\n  Sample orphan di det_sales:")
    for r in list(orphan_det)[:5]:
        print(f"    {r}")


CEK 1: NO_RESEP — det_sales <-> ms_sales
  NO_RESEP unik di ms_sales  :    127,064
  NO_RESEP unik di det_sales :    157,704
  Orphan di det_sales        :     30,750  <- tidak ada header di ms_sales
  Orphan di ms_sales         :        110  <- tidak ada detail di det_sales

  Sample orphan di det_sales:
    RJ-01.2015-40-0578
    RI-02.2015-33-0793
    RJ-02.2015-40-0966
    RJ-02.2015-40-2145
    KC-01.2015-09-0012


In [11]:
# ── CEK 2: KD_OBAT — det_sales/transaction <-> ms_product ──────────────────
assert ms_product  is not None, "ms_product belum ter-parse"
assert transaction is not None, "transaction belum ter-parse"

print("=" * 64)
print("CEK 2: KD_OBAT — det_sales / transaction <-> ms_product")
print("=" * 64)

kd_product = set(ms_product['KD_OBAT'].dropna().unique())
kd_det     = set(det_sales['KD_OBAT'].dropna().unique())
kd_trans   = set(transaction['KD_OBAT'].dropna().unique())

orphan_det_kd   = kd_det  - kd_product
orphan_trans_kd = kd_trans - kd_product

print(f"  KD_OBAT unik di ms_product  : {len(kd_product):>8,}")
print(f"  KD_OBAT unik di det_sales   : {len(kd_det):>8,}")
print(f"  KD_OBAT unik di transaction : {len(kd_trans):>8,}")
print()
print(f"  Orphan KD_OBAT di det_sales   : {len(orphan_det_kd):>6,}  <- tidak di ms_product")
print(f"  Orphan KD_OBAT di transaction : {len(orphan_trans_kd):>6,}  <- tidak di ms_product")

if orphan_det_kd:
    print(f"\n  Sample orphan KD_OBAT di det_sales:")
    for k in list(orphan_det_kd)[:5]:
        print(f"    {k}")


CEK 2: KD_OBAT — det_sales / transaction <-> ms_product
  KD_OBAT unik di ms_product  :    6,878
  KD_OBAT unik di det_sales   :    2,233
  KD_OBAT unik di transaction :    2,043

  Orphan KD_OBAT di det_sales   :      6  <- tidak di ms_product
  Orphan KD_OBAT di transaction :      6  <- tidak di ms_product

  Sample orphan KD_OBAT di det_sales:
    AI-0954
    AI-1539
    R-4681
    R-2827
    R-1848


---
## TAHAP 2 — Pembersihan & Transformasi Data (Data Cleaning)

Pada tahap ini kita akan:
- Mengonversi tipe data kolom numerik dan tanggal ke format yang benar.
- Menandai anomali data (Qty negatif sebagai retur, produk dengan harga jual 0).
- Menggabungkan (*join*) data transaksi dengan master produk dan master sales header.
- Membuat kolom turunan untuk dimensi waktu, klasifikasi jenis transaksi, dan profitabilitas.
- Mengekspor dataset bersih ke format Parquet dan CSV.

In [12]:
# ── 2.1 Pembersihan & Konversi Tipe Data ──────────────────────────────────────
print("Melakukan konversi tipe data...")

# det_sales
det_sales_clean = det_sales.copy()
det_sales_clean['QTY'] = pd.to_numeric(det_sales_clean['QTY'], errors='coerce').fillna(0).astype(int)
det_sales_clean['HNA'] = pd.to_numeric(det_sales_clean['HNA'], errors='coerce').fillna(0.0).astype(float)
det_sales_clean['HJ'] = pd.to_numeric(det_sales_clean['HJ'], errors='coerce').fillna(0.0).astype(float)
det_sales_clean['PPN_JUAL'] = pd.to_numeric(det_sales_clean['PPN_JUAL'], errors='coerce').fillna(0.0).astype(float)

# ms_product
ms_product_clean = ms_product.copy()
ms_product_clean['HJ_RP'] = pd.to_numeric(ms_product_clean['HJ_RP'], errors='coerce').fillna(0.0).astype(float)

# ms_sales
ms_sales_clean = ms_sales.copy()
ms_sales_clean['TGL'] = pd.to_datetime(ms_sales_clean['TGL'], errors='coerce')

print("Tipe data setelah dikonversi:")
print("det_sales:\n", det_sales_clean.dtypes)
print("\nms_product:\n", ms_product_clean.dtypes)
print("\nms_sales:\n", ms_sales_clean.dtypes)

Melakukan konversi tipe data...
Tipe data setelah dikonversi:
det_sales:
 NO_RESEP     object
KD_OBAT      object
QTY           int64
HNA         float64
HJ          float64
PPN_JUAL    float64
dtype: object

ms_product:
 KD_OBAT       object
NAMA          object
SAT_JUAL      object
KD_PABRIK     object
HJ_RP        float64
dtype: object

ms_sales:
 NO_RESEP             object
TGL          datetime64[ns]
KD_CUST              object
KD_DOKTER            object
REG_AS               object
JAM_JUAL             object
RACIK                object
dtype: object


In [13]:
# ── 2.2 Penanganan Anomali & Penandaan (Flags) ──────────────────────────────
# Menandai transaksi retur
det_sales_clean['is_return'] = det_sales_clean['QTY'] < 0

# Menandai produk dengan harga 0
ms_product_clean['is_zero_price'] = ms_product_clean['HJ_RP'] == 0.0

print(f"Jumlah transaksi retur (det_sales)           : {det_sales_clean['is_return'].sum():,}")
print(f"Jumlah produk dengan harga jual 0 (ms_product): {ms_product_clean['is_zero_price'].sum():,}")

Jumlah transaksi retur (det_sales)           : 277
Jumlah produk dengan harga jual 0 (ms_product): 859


In [14]:
# ── 2.3 Denormalisasi (Merge/Join) ──────────────────────────────────────────
# Step 1: Join det_sales dengan ms_product
df_sales = pd.merge(
    det_sales_clean,
    ms_product_clean[['KD_OBAT', 'NAMA', 'SAT_JUAL', 'KD_PABRIK']],
    on='KD_OBAT',
    how='left'
)

# Step 2: Join dengan ms_sales
df_sales = pd.merge(
    df_sales,
    ms_sales_clean[['NO_RESEP', 'TGL', 'KD_CUST', 'KD_DOKTER', 'REG_AS', 'JAM_JUAL', 'RACIK']],
    on='NO_RESEP',
    how='left'
)

# Tandai resep yatim (orphan)
df_sales['is_orphan'] = df_sales['TGL'].isna()

print(f"Shape dataset hasil gabungan: {df_sales.shape[0]:,} baris x {df_sales.shape[1]} kolom")
print(f"Jumlah baris orphan (resep yatim): {df_sales['is_orphan'].sum():,} ({df_sales['is_orphan'].mean()*100:.1f}%)")

Shape dataset hasil gabungan: 514,620 baris x 17 kolom
Jumlah baris orphan (resep yatim): 102,605 (19.9%)


In [15]:
# ── 2.4 Pembuatan Kolom Turunan (Transformasi) ──────────────────────────────
print("Membuat kolom turunan...")

# Dimensi Waktu (hanya untuk data non-orphan)
df_sales['Tahun'] = df_sales['TGL'].dt.year
df_sales['Bulan'] = df_sales['TGL'].dt.month
df_sales['Hari'] = df_sales['TGL'].dt.day
df_sales['Nama_Hari'] = df_sales['TGL'].dt.day_name()

# Ekstrak Jam dari JAM_JUAL
df_sales['Jam'] = pd.to_numeric(df_sales['JAM_JUAL'].str.split(':').str[0], errors='coerce')

# Klasifikasi Jenis Transaksi dari prefix NO_RESEP
extracted_prefix = df_sales['NO_RESEP'].str.extract(r'^([A-Z]+(?:/[A-Z]+)?(?:-\d+)?)')[0]
base_prefix = extracted_prefix.str.extract(r'^([A-Z]+)')[0]

prefix_map = {
    'RJ': 'Rawat Jalan',
    'RI': 'Rawat Inap',
    'RL': 'Resep Luar / Unit Lain',
    'KC': 'Kerjasama / Karyawan',
    'B' : 'Obat Bebas',
    'PR': 'Lainnya (PR)'
}
df_sales['jenis_transaksi'] = base_prefix.map(prefix_map).fillna('Lainnya')

# Terjemahan Nama Hari ke Indonesia
indo_days = {
    'Monday': 'Senin', 'Tuesday': 'Selasa', 'Wednesday': 'Rabu',
    'Thursday': 'Kamis', 'Friday': 'Jumat', 'Saturday': 'Sabtu', 'Sunday': 'Minggu'
}
df_sales['Nama_Hari'] = df_sales['Nama_Hari'].map(indo_days)

# Metrik Keuangan
df_sales['TOTAL_HJ'] = df_sales['QTY'] * df_sales['HJ']
df_sales['TOTAL_HNA'] = df_sales['QTY'] * df_sales['HNA']
df_sales['MARGIN'] = df_sales['HJ'] - df_sales['HNA']
df_sales['TOTAL_MARGIN'] = df_sales['TOTAL_HJ'] - df_sales['TOTAL_HNA']

# Menghitung MARGIN_PCT secara aman (vectorized)
df_sales['MARGIN_PCT'] = 0.0
mask = df_sales['TOTAL_HNA'] > 0
df_sales.loc[mask, 'MARGIN_PCT'] = df_sales.loc[mask, 'TOTAL_MARGIN'] / df_sales.loc[mask, 'TOTAL_HNA']

print("Kolom turunan sukses dibuat.")
print("\nPreview data baru (3 baris):")
print(df_sales[['NO_RESEP', 'TGL', 'jenis_transaksi', 'Nama_Hari', 'QTY', 'TOTAL_HJ', 'TOTAL_MARGIN']].head(3))

Membuat kolom turunan...
Kolom turunan sukses dibuat.

Preview data baru (3 baris):
             NO_RESEP        TGL jenis_transaksi Nama_Hari  QTY  TOTAL_HJ  \
0  RJ-01.2015-08-0001 2015-01-02     Rawat Jalan     Jumat    1  1,615.35   
1  RJ-01.2015-08-0001 2015-01-02     Rawat Jalan     Jumat    2    934.12   
2  RJ-01.2015-08-0001 2015-01-02     Rawat Jalan     Jumat    3    297.66   

   TOTAL_MARGIN  
0        280.35  
1        162.12  
2         51.66  


In [16]:
# ── 2.5 Validasi & Ekspor Dataset Bersih ─────────────────────────────────────
# Cek null pada kolom kunci non-orphan
non_orphan = df_sales[~df_sales['is_orphan']]
print("Pengecekan Nilai Null (Data Non-Orphan):")
for col in ['NO_RESEP', 'KD_OBAT', 'QTY', 'TGL']:
    print(f"  Jumlah null pada '{col}': {non_orphan[col].isna().sum()}")

# Cek outlier ekstrem (QTY > 1000)
outliers = df_sales[df_sales['QTY'].abs() > 1000]
print(f"\nJumlah baris dengan QTY ekstrem (|QTY| > 1000): {len(outliers)}")
if len(outliers) > 0:
    print(outliers[['NO_RESEP', 'TGL', 'NAMA', 'QTY', 'TOTAL_HJ']])

# Ekspor ke Parquet
parquet_path = OUTPUT_DIR / 'sales_clean.parquet'
df_sales.to_parquet(parquet_path, index=False)
print(f"\n[OK] Dataset disimpan ke Parquet: {parquet_path} ({parquet_path.stat().st_size/(1024**2):.2f} MB)")

# Ekspor ke CSV (compressed gzip)
csv_path = OUTPUT_DIR / 'sales_clean.csv.gz'
df_sales.to_csv(csv_path, index=False, compression='gzip')
print(f"[OK] Dataset disimpan ke CSV (Gzip): {csv_path} ({csv_path.stat().st_size/(1024**2):.2f} MB)")

Pengecekan Nilai Null (Data Non-Orphan):
  Jumlah null pada 'NO_RESEP': 0
  Jumlah null pada 'KD_OBAT': 0
  Jumlah null pada 'QTY': 0
  Jumlah null pada 'TGL': 0

Jumlah baris dengan QTY ekstrem (|QTY| > 1000): 54
                  NO_RESEP        TGL                          NAMA    QTY  \
56878   RJ-03.2015-37-0033 2015-03-07                  Becefort tab  12000   
199297  RJ-07.2015-03-0001 2015-07-14            Amoxicillin 500 mg   1440   
199298  RJ-07.2015-03-0001 2015-07-14               Tetracyline 250   1440   
271556  RJ-09.2015-03-0024 2015-09-18            Amoxicillin 500 mg  -1996   
271559  RJ-09.2015-03-0024 2015-09-18      Asam Mefenamat 500mg tab  -1390   
271638  RJ-09.2015-03-0023 2015-09-18      Asam Mefenamat 500mg tab   4400   
271639  RJ-09.2015-03-0023 2015-09-18            Amoxicillin 500 mg   4000   
271649  RJ-09.2015-03-0023 2015-09-18                       FLU TAB   1560   
271651  RJ-09.2015-03-0023 2015-09-18              Neurodex FC komb   3000   
271661

---
## TAHAP 3 — Eksplorasi Data (EDA) & Ringkasan Awal

Pada tahap ini kita akan mengeksplorasi data penjualan bersih 2015 untuk menghasilkan:
- Statistik deskriptif utama (Resep unik, item terjual, omzet, total margin profit).
- Ringkasan bulanan untuk melihat tren omzet.
- Distribusi transaksi berdasarkan jenis transaksi, jam operasional, dan hari.
- Analisis obat terlaris berdasarkan volume (QTY), nilai omzet, margin keuntungan, serta produsen/pabrik.
- Profil dokter teraktif, perbandingan asuransi (Reguler vs Askes), dan resep racikan vs non-racikan.

In [17]:
# ── 3.1 Statistik Deskriptif Utama & Penjualan Bulanan ─────────────────────
# Menggunakan data normal (non-orphan & non-return) untuk analisis tren penjualan normal
df_normal = df_sales[(~df_sales['is_orphan']) & (~df_sales['is_return'])].copy()

n_resep = df_normal['NO_RESEP'].nunique()
n_items = df_normal['QTY'].sum()
total_omzet = df_normal['TOTAL_HJ'].sum()
total_hna = df_normal['TOTAL_HNA'].sum()
total_margin = df_normal['TOTAL_MARGIN'].sum()
margin_pct = total_margin / total_hna * 100 if total_hna > 0 else 0

print("=" * 64)
print("  STATISTIK KUNCI PENJUALAN APOTEK 2015 (NORMAL & NON-ORPHAN)")
print("=" * 64)
print(f"  Total Resep Unik     : {n_resep:>15,}")
print(f"  Total Item Terjual   : {n_items:>15,}")
print(f"  Total Omzet (Kotor)  : Rp {total_omzet:>12,.2f}")
print(f"  Total COGS (HNA)     : Rp {total_hna:>12,.2f}")
print(f"  Total Margin Profit  : Rp {total_margin:>12,.2f}")
print(f"  Persentase Margin    : {margin_pct:>14.2f}%")
print("=" * 64)

# Ringkasan Bulanan
df_normal['Bulan_Nama'] = df_normal['TGL'].dt.strftime('%m - %B')
indo_months = {
    '01 - January': '01 - Januari', '02 - February': '02 - Februari', '03 - March': '03 - Maret',
    '04 - April': '04 - April', '05 - May': '05 - Mei', '06 - June': '06 - Juni',
    '07 - July': '07 - Juli', '08 - August': '08 - Agustus', '09 - September': '09 - September',
    '10 - October': '10 - Oktober', '11 - November': '11 - November', '12 - December': '12 - Desember'
}
df_normal['Bulan_Nama'] = df_normal['Bulan_Nama'].map(indo_months)

monthly_summary = df_normal.groupby('Bulan_Nama').agg(
    jumlah_resep=('NO_RESEP', 'nunique'),
    qty_terjual=('QTY', 'sum'),
    omzet=('TOTAL_HJ', 'sum'),
    margin=('TOTAL_MARGIN', 'sum')
).reset_index()

monthly_summary['margin_pct'] = (monthly_summary['margin'] / (monthly_summary['omzet'] - monthly_summary['margin'])) * 100
print("\nRingkasan Kinerja Penjualan Bulanan:")
print(monthly_summary.to_string(index=False, formatters={
    'jumlah_resep': '{:,.0f}'.format,
    'qty_terjual': '{:,.0f}'.format,
    'omzet': 'Rp {:,.2f}'.format,
    'margin': 'Rp {:,.2f}'.format,
    'margin_pct': '{:.2f}%'.format
}))

  STATISTIK KUNCI PENJUALAN APOTEK 2015 (NORMAL & NON-ORPHAN)
  Total Resep Unik     :         126,932
  Total Item Terjual   :       4,127,902
  Total Omzet (Kotor)  : Rp 15,305,738,579.84
  Total COGS (HNA)     : Rp 11,492,528,466.25
  Total Margin Profit  : Rp 3,813,210,113.59
  Persentase Margin    :          33.18%

Ringkasan Kinerja Penjualan Bulanan:
    Bulan_Nama jumlah_resep qty_terjual               omzet            margin margin_pct
  01 - Januari        7,231     127,317   Rp 663,048,120.45 Rp 172,960,561.03     35.29%
 02 - Februari        9,589     276,694   Rp 955,002,659.17 Rp 226,326,581.44     31.06%
    03 - Maret       10,277     324,477 Rp 1,296,809,012.98 Rp 321,085,039.43     32.91%
    04 - April       10,740     317,725 Rp 1,247,735,123.12 Rp 314,126,454.21     33.65%
      05 - Mei        9,858     287,722 Rp 1,211,530,057.86 Rp 309,248,407.59     34.27%
     06 - Juni       11,511     353,070 Rp 1,275,534,846.74 Rp 317,277,595.74     33.11%
     07 - Juli   

In [18]:
# ── 3.2 Analisis Jenis Transaksi ──────────────────────────────────────────
# Kita menggunakan seluruh data non-orphan untuk membandingkan proporsi transaksi
df_analysis = df_sales[~df_sales['is_orphan']].copy()

tx_summary = df_analysis.groupby('jenis_transaksi').agg(
    jumlah_resep=('NO_RESEP', 'nunique'),
    omzet=('TOTAL_HJ', 'sum'),
    margin=('TOTAL_MARGIN', 'sum')
).reset_index()

tx_summary['omzet_pct'] = tx_summary['omzet'] / tx_summary['omzet'].sum() * 100
tx_summary['margin_pct'] = tx_summary['margin'] / tx_summary['margin'].sum() * 100

print("Analisis Berdasarkan Jenis Transaksi:")
print(tx_summary.to_string(index=False, formatters={
    'jumlah_resep': '{:,.0f}'.format,
    'omzet': 'Rp {:,.2f}'.format,
    'margin': 'Rp {:,.2f}'.format,
    'omzet_pct': '{:.2f}%'.format,
    'margin_pct': '{:.2f}%'.format
}))

Analisis Berdasarkan Jenis Transaksi:
       jenis_transaksi jumlah_resep               omzet              margin omzet_pct margin_pct
  Kerjasama / Karyawan        2,349   Rp 137,066,634.26    Rp 23,644,276.26     0.90%      0.62%
          Lainnya (PR)            8     Rp 1,453,033.00       Rp 341,818.00     0.01%      0.01%
            Obat Bebas           87     Rp 6,819,170.00     Rp 1,840,832.00     0.04%      0.05%
            Rawat Inap       44,834 Rp 4,515,453,488.22 Rp 1,121,308,491.86    29.58%     29.47%
           Rawat Jalan       71,941 Rp 9,641,765,536.35 Rp 2,403,614,233.46    63.15%     63.18%
Resep Luar / Unit Lain        7,735   Rp 965,079,745.62   Rp 253,569,495.62     6.32%      6.67%


In [19]:
# ── 3.3 Analisis Tren Waktu ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# 1. Tren Omzet Bulanan (Ubah ke Line Chart dengan Area Fill)
plt.figure()
sns.lineplot(data=monthly_summary, x='Bulan_Nama', y='omzet', marker='o', color='teal', linewidth=3, markersize=8)
# Tambahkan fill area di bawah garis untuk estetika visual yang lebih premium
plt.fill_between(monthly_summary['Bulan_Nama'], monthly_summary['omzet'], color='teal', alpha=0.15)
plt.title('Tren Omzet Penjualan Bulanan (2015)', fontsize=14, fontweight='bold')
plt.xlabel('Bulan')
plt.ylabel('Total Omzet (Rupiah)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tren_omzet_bulanan.png', dpi=150)
plt.close()

# 2. Distribusi Jam Sibuk
jam_summary = df_normal.groupby('Jam').agg(
    jumlah_resep=('NO_RESEP', 'nunique')
).reset_index()

plt.figure()
sns.lineplot(data=jam_summary, x='Jam', y='jumlah_resep', marker='o', color='crimson', linewidth=2)
plt.title('Distribusi Waktu Transaksi (Jam Sibuk Pelayanan Apotek)', fontsize=14, fontweight='bold')
plt.xlabel('Jam Operasional')
plt.ylabel('Jumlah Resep')
plt.xticks(range(24))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'jam_sibuk_pelayanan.png', dpi=150)
plt.close()

# 3. Hari Sibuk dalam Seminggu
hari_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
hari_summary = df_normal.groupby('Nama_Hari').agg(
    jumlah_resep=('NO_RESEP', 'nunique')
).reindex(hari_order).reset_index()

plt.figure()
sns.barplot(data=hari_summary, x='Nama_Hari', y='jumlah_resep', palette='Blues_r')
plt.title('Hari Tersibuk Pelayanan Apotek dalam Seminggu', fontsize=14, fontweight='bold')
plt.xlabel('Hari')
plt.ylabel('Jumlah Resep')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hari_sibuk.png', dpi=150)
plt.close()

print("[OK] Tiga grafik visualisasi tren waktu disimpan di direktori 'output/':")
print("  - tren_omzet_bulanan.png (Telah diubah menjadi Line Chart)")
print("  - jam_sibuk_pelayanan.png")
print("  - hari_sibuk.png")

[OK] Tiga grafik visualisasi tren waktu disimpan di direktori 'output/':
  - tren_omzet_bulanan.png (Telah diubah menjadi Line Chart)
  - jam_sibuk_pelayanan.png
  - hari_sibuk.png


In [20]:
# ── 3.4 Analisis Produk Terlaris ──────────────────────────────────────────
# Kita menggunakan seluruh data (termasuk orphan) untuk performa produk agregat

# Top 20 Volume (QTY)
top_qty = df_sales.groupby(['KD_OBAT', 'NAMA']).agg(
    total_qty=('QTY', 'sum'),
    total_omzet=('TOTAL_HJ', 'sum')
).sort_values('total_qty', ascending=False).head(20).reset_index()

print("=" * 64)
print("  TOP 20 PRODUK OBAT BERDASARKAN VOLUME PENJUALAN (QTY)")
print("=" * 64)
print(top_qty.to_string(index=False, formatters={
    'total_qty': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format
}))

# Top 20 Omzet (TOTAL_HJ)
top_omzet = df_sales.groupby(['KD_OBAT', 'NAMA']).agg(
    total_qty=('QTY', 'sum'),
    total_omzet=('TOTAL_HJ', 'sum')
).sort_values('total_omzet', ascending=False).head(20).reset_index()

print("\n" + "=" * 64)
print("  TOP 20 PRODUK OBAT BERDASARKAN KONTRIBUSI OMZET (TOTAL_HJ)")
print("=" * 64)
print(top_omzet.to_string(index=False, formatters={
    'total_qty': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format
}))

# Top 20 Total Margin (TOTAL_MARGIN)
top_margin = df_sales.groupby(['KD_OBAT', 'NAMA']).agg(
    total_qty=('QTY', 'sum'),
    total_margin=('TOTAL_MARGIN', 'sum')
).sort_values('total_margin', ascending=False).head(20).reset_index()

print("\n" + "=" * 64)
print("  TOP 20 PRODUK OBAT BERDASARKAN TOTAL MARGIN PROFIT")
print("=" * 64)
print(top_margin.to_string(index=False, formatters={
    'total_qty': '{:,.0f}'.format,
    'total_margin': 'Rp {:,.2f}'.format
}))

# Distribusi Penjualan Per Pabrik
pabrik_summary = df_sales.groupby('KD_PABRIK').agg(
    total_qty=('QTY', 'sum'),
    total_omzet=('TOTAL_HJ', 'sum'),
    total_margin=('TOTAL_MARGIN', 'sum')
).sort_values('total_omzet', ascending=False).head(15).reset_index()

print("\n" + "=" * 64)
print("  DISTRIBUSI PENJUALAN PER PABRIK (TOP 15)")
print("=" * 64)
print(pabrik_summary.to_string(index=False, formatters={
    'total_qty': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format,
    'total_margin': 'Rp {:,.2f}'.format
}))

  TOP 20 PRODUK OBAT BERDASARKAN VOLUME PENJUALAN (QTY)
KD_OBAT                       NAMA total_qty       total_omzet
AI-0853           Metformin 500 mg   173,131  Rp 24,727,794.88
AI-0943           Neurodex FC komb   167,570  Rp 47,678,674.02
AI-0190        Candesartan TI 8 mg   160,495 Rp 652,803,605.82
AI-0958 Nitrokaf Retard Forte 5 mg   138,153 Rp 384,259,032.65
AI-0406                Eclid 50 mg   133,644 Rp 148,531,250.41
AI-0771         Lansoprazole 30 mg   132,443  Rp 87,974,693.61
AI-0097               Aptor 100 mg    99,927  Rp 19,058,148.80
AI-0860    Methylprednisolone 4 mg    89,163  Rp 26,874,579.19
AI-0068             Amlodipin 5 mg    80,922  Rp 28,635,644.34
AI-0106      Asam Mefenamat 500 mg    73,323  Rp 10,457,816.81
AI-0043         Allopurinol 100 mg    70,894   Rp 8,252,914.46
AI-0067            Amlodipin 10 mg    66,755  Rp 40,726,755.54
AI-1168            Renadinac 25 mg    57,871   Rp 9,312,030.23
AI-1028            Osteocal 500 mg    57,700  Rp 39,339,012.45

In [21]:
# ── 3.5 Analisis Dokter & Pelanggan ───────────────────────────────────────
# Analisis Dokter teraktif
df_non_orphan = df_sales[~df_sales['is_orphan']].copy()

dokter_summary = df_non_orphan.groupby('KD_DOKTER').agg(
    jumlah_resep=('NO_RESEP', 'nunique'),
    total_omzet=('TOTAL_HJ', 'sum')
).sort_values('jumlah_resep', ascending=False).head(15).reset_index()

dokter_summary['omzet_per_resep'] = dokter_summary['total_omzet'] / dokter_summary['jumlah_resep']

print("Top 15 Dokter Berdasarkan Penulisan Resep Terbanyak:")
print(dokter_summary.to_string(index=False, formatters={
    'jumlah_resep': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format,
    'omzet_per_resep': 'Rp {:,.2f}'.format
}))

# Analisis Reguler vs Askes
asuransi_summary = df_non_orphan.groupby('REG_AS').agg(
    jumlah_resep=('NO_RESEP', 'nunique'),
    total_omzet=('TOTAL_HJ', 'sum'),
    total_margin=('TOTAL_MARGIN', 'sum')
).reset_index()

print("\nPerbandingan Penjualan Reguler vs Askes:")
print(asuransi_summary.to_string(index=False, formatters={
    'jumlah_resep': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format,
    'total_margin': 'Rp {:,.2f}'.format
}))

# Perbandingan Racikan vs Non-Racikan
racik_summary = df_non_orphan.groupby('RACIK').agg(
    jumlah_resep=('NO_RESEP', 'nunique'),
    total_omzet=('TOTAL_HJ', 'sum')
).reset_index()
racik_summary['omzet_per_resep'] = racik_summary['total_omzet'] / racik_summary['jumlah_resep']

print("\nPerbandingan Resep Racikan (Y) vs Non-Racikan (N):")
print(racik_summary.to_string(index=False, formatters={
    'jumlah_resep': '{:,.0f}'.format,
    'total_omzet': 'Rp {:,.2f}'.format,
    'omzet_per_resep': 'Rp {:,.2f}'.format
}))

Top 15 Dokter Berdasarkan Penulisan Resep Terbanyak:
KD_DOKTER jumlah_resep         total_omzet omzet_per_resep
      001       43,289 Rp 6,752,744,764.40   Rp 155,992.16
      000       28,467 Rp 2,208,657,542.56    Rp 77,586.59
      003        7,175   Rp 756,105,347.58   Rp 105,380.54
      002        5,751   Rp 554,405,057.20    Rp 96,401.51
      004        2,637   Rp 154,368,792.40    Rp 58,539.55
      017        2,509   Rp 337,380,606.34   Rp 134,468.16
      028        2,453   Rp 170,550,997.47    Rp 69,527.52
      019        2,037   Rp 958,045,396.96   Rp 470,321.75
      031        1,929   Rp 232,290,029.65   Rp 120,419.92
      008        1,749   Rp 392,511,344.99   Rp 224,420.44
      062        1,688   Rp 363,438,878.18   Rp 215,307.39
      040        1,650    Rp 18,415,117.70    Rp 11,160.68
      045        1,632   Rp 130,122,722.25    Rp 79,732.06
      027        1,480    Rp 44,286,980.95    Rp 29,923.64
      016        1,257   Rp 108,091,113.64    Rp 85,991.34

Pe

In [22]:
# ── 3.6 Penyimpanan Ringkasan Akhir ───────────────────────────────────────
import json

# Menyiapkan ringkasan JSON
eda_summary = {
    "total_omzet_kotor": float(total_omzet),
    "total_cogs": float(total_hna),
    "total_margin_keuntungan": float(total_margin),
    "persentase_margin_total": float(margin_pct),
    "jumlah_resep_unik": int(n_resep),
    "jumlah_item_terjual": int(n_items),
    
    "kinerja_bulanan": monthly_summary.to_dict(orient='records'),
    "kinerja_jenis_transaksi": tx_summary.to_dict(orient='records'),
    
    "top_5_produk_qty": top_qty.head(5)[['KD_OBAT', 'NAMA', 'total_qty', 'total_omzet']].to_dict(orient='records'),
    "top_5_produk_omzet": top_omzet.head(5)[['KD_OBAT', 'NAMA', 'total_qty', 'total_omzet']].to_dict(orient='records'),
    
    "dokter_teraktif": dokter_summary.head(5).to_dict(orient='records'),
    "porsi_asuransi": asuransi_summary.to_dict(orient='records'),
    "porsi_racik": racik_summary.to_dict(orient='records')
}

summary_file = OUTPUT_DIR / 'tahap3_eda_summary.json'
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(eda_summary, f, indent=2, ensure_ascii=False)

print(f"[OK] Ringkasan EDA disimpan ke: {summary_file}")

[OK] Ringkasan EDA disimpan ke: d:\PORTOFOLIO\sales\output\tahap3_eda_summary.json
